In [2]:
from dotenv import load_dotenv
import os

langsmith_key = os.getenv("LANGCHAIN_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_API_KEY"]=langsmith_key
os.environ["LANGCHAIN_PROJECT"]="langchain-course"

In [3]:
api_key = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = api_key

api_version = os.getenv("OPENAI_API_VERSION")
os.environ["OPENAI_API_VERSION"] = api_version

api_endpoint = os.getenv("OPENAI_AZURE_ENDPOINT")
os.environ["OPENAI_AZURE_ENDPOINT"] = api_endpoint

api_model = os.getenv("OPENAI_AZURE_MODEL")
os.environ["OPENAI_AZURE_MODEL"] = api_model


In [4]:
from langchain_openai import AzureChatOpenAI

llm=AzureChatOpenAI(
        api_key= os.environ["OPENAI_API_KEY"],
        api_version= os.environ["OPENAI_API_VERSION"],
        azure_endpoint= os.environ["OPENAI_AZURE_ENDPOINT"],
        model_name= os.environ["OPENAI_AZURE_MODEL"],
        temperature=0.4,
        max_tokens=1000,
        seed=42,
)
llm_response=llm.invoke("Tell me a joke")
llm_response

AIMessage(content='Why did the scarecrow win an award? \n\nBecause he was outstanding in his field!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 11, 'total_tokens': 30, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'id': 'chatcmpl-CQmkra2GnzTHFuxAhQJwx6ZcV42nT', 'service_tier': None, 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}], 'finish_reason': 'stop', 'logprobs': None, 'content_filter_results': 

In [5]:
from langchain_core.output_parsers import StrOutputParser
output_parser=StrOutputParser()
output_parser.invoke(llm_response)

'Why did the scarecrow win an award? \n\nBecause he was outstanding in his field!'

In [6]:
chain= llm | output_parser
chain.invoke('tell me a joke')

'Why did the scarecrow win an award? \n\nBecause he was outstanding in his field!'

In [7]:
from typing import List
from pydantic import BaseModel, Field

class MobileReview(BaseModel):
    phone_model: str = Field(description="Name and model of the phone")
    rating: float= Field(description="Ovarall rating out of 5")
    pros: List[str]= Field(description="List of positive aspects")
    cons: List[str] =  Field(description="List of negative aspects")
    summary: str=Field(description="Brief summary of the review")

review_text="""
If texting is your main mode of communication, the Pixel 8 is a near-perfect companion. The keyboard feels responsive and accurate — Google’s Gboard prediction is smarter than ever, often finishing your sentences correctly. Haptic feedback is subtle but satisfying, giving each keypress a sense of precision.

Notifications are neatly organized, and the quick-reply feature works flawlessly from the lock screen. You can keep up with group chats or DMs without even unlocking your phone — a real time-saver. Battery life easily stretches past a day, even with constant messaging, photos, and GIF sharing.

However, the small downside is the occasional lag when switching between chat apps, especially after long uptime, and the emoji picker still takes too many taps.

Overall, the Pixel 8 feels designed for heavy texters — fast, smooth, smart, and dependable. If you live in your chat apps, this one won’t disappoint.
"""

structured_llm=llm.with_structured_output(MobileReview)
output=structured_llm.invoke(review_text)
output

MobileReview(phone_model='Google Pixel 8', rating=4.5, pros=['Responsive and accurate keyboard', 'Smart Gboard predictions', 'Subtle and satisfying haptic feedback', 'Organized notifications', 'Flawless quick-reply feature from lock screen', 'Excellent battery life for heavy usage'], cons=['Occasional lag when switching between chat apps', 'Emoji picker requires too many taps'], summary='The Google Pixel 8 is an excellent choice for heavy texters, offering a responsive keyboard, smart predictions, and great battery life, despite minor lags in app switching.')

In [8]:
output.pros

['Responsive and accurate keyboard',
 'Smart Gboard predictions',
 'Subtle and satisfying haptic feedback',
 'Organized notifications',
 'Flawless quick-reply feature from lock screen',
 'Excellent battery life for heavy usage']

Prompt Template

In [9]:
from langchain_core.prompts import ChatPromptTemplate
prompt= ChatPromptTemplate.from_template("Tell me short joke about {topic}")
prompt.invoke({"topic": "programming"})

ChatPromptValue(messages=[HumanMessage(content='Tell me short joke about programming', additional_kwargs={}, response_metadata={})])

In [10]:
chain= prompt | llm | output_parser
chain.invoke({"topic":"car driver"})

'Why did the car driver bring a ladder?\n\nBecause they wanted to reach new heights in driving!'

LLM Messages

In [11]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage

system_message= SystemMessage(content="You are a helpful assistant that tells jokes")
human_message= HumanMessage(content="Tell me about programming")
llm.invoke([system_message, human_message])

AIMessage(content="Sure! Programming is like telling a computer what to do, but instead of using spoken language, you use a programming language. It's a bit like giving a recipe to a chef, but instead of cooking food, the computer performs tasks.\n\nAnd speaking of programming, here's a joke for you:\n\nWhy do programmers prefer dark mode?\n\nBecause light attracts bugs!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 23, 'total_tokens': 94, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'id': 'chatcmpl-CQml2Gsuor3996aM70HKg73FQlJ1A', 'service_tier': None, 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbrea

In [12]:
template=ChatPromptTemplate([
    ("system","You are a helpfull assistance that tells jokes"),
    ("human", "Tell me about {topic}")
])

prompt_value=template.invoke({"topic":"programming"})
llm.invoke(prompt_value)


AIMessage(content="Sure! Programming is the process of creating a set of instructions that a computer can follow to perform specific tasks. It's like giving a computer a recipe to follow. \n\nNow, here’s a programming joke for you:\n\nWhy do programmers prefer dark mode?\n\nBecause light attracts bugs!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 24, 'total_tokens': 80, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'id': 'chatcmpl-CQml33dsgAK6swTyJRfBTJXrKNpIg', 'service_tier': None, 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': 

In [13]:
! pip install docx2txt 

1: Load Documents and Create chunks of it

In [1]:
from langchain_community. document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from typing import List
from langchain_core.documents import Document

text_spliter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

docx_loder=Docx2txtLoader(r"C:\Users\HP\Desktop\Langchain\RAG\OfflineData\HR documents.docx", )
documents=docx_loder.load()
print(len(documents))
splits=text_spliter.split_documents(documents)
print(f"Splits the documents into {len(splits)} chunks.")

1
Splits the documents into 5 chunks.


In [3]:
splits

[Document(metadata={'source': 'C:\\Users\\HP\\Desktop\\Langchain\\RAG\\OfflineData\\HR documents.docx'}, page_content='[Document ID: KB001]\n\nTitle: Unable to connect to VPN\n\nContent:\n\nIf users cannot connect to VPN, verify their credentials, check if the VPN service is running, and ensure network ports 443 and 500 are open. If the issue persists, restart the VPN client or check the firewall settings.\n\n\n\n---\n\n\n\n[Document ID: KB002]\n\nTitle: Resetting Active Directory Password\n\nContent:\n\nTo reset a user’s Active Directory password, open Active Directory Users and Computers, right-click the user account, and select "Reset Password." Ensure “User must change password at next logon” is checked for compliance.\n\n\n\n---\n\n\n\n[Document ID: KB003]\n\nTitle: DNS Resolution Issue\n\nContent:\n\nIf DNS resolution fails, verify the DNS server IP configuration in network adapter settings. Use the nslookup command to test name resolution. Restart the DNS client service or flush

In [15]:
# function to load documents from a folder
import os
def load_documents(folder_path:str)-> List[Document]:
    documents=[]
    for filename in os.listdir(folder_path):
        file_path=os.path.join(folder_path, filename)
        # if filename.endswith('.pdf'):
        #     loader=PyPDFLoader(file_path)
        if filename.endswith('.docx'):
            loader=Docx2txtLoader(file_path)
        else:
            print(f"Unsupported file type: {filename}")
            continue
        documents.extend(loader.load())
    return documents

# Load documents from a folder
folder_path="C:\\Users\\HP\\Desktop\\Langchain\\RAG\\OfflineData"
documents= load_documents(folder_path)

print(f" Loaded {len(documents)} documents from the folder")
splits=text_spliter.split_documents(documents)
print(f"Split the documents into {len(splits)} chunks")


Unsupported file type: company_data.csv
Unsupported file type: DSA.pdf
Unsupported file type: IT_Knowledge_Base.pdf
Unsupported file type: SQL.pdf
Unsupported file type: wipro.txt
 Loaded 1 documents from the folder
Split the documents into 5 chunks


2: Create Embeddings of the chunks

In [ ]:
#! pip install sentence_transformers
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
embedding_function=SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
document_embeddings=embedding_function.embed_documents([split.page_content for split in splits])
document_embeddings[0]

3: Create and persist Chroma vectore store

In [17]:
from langchain_chroma import Chroma

embedding_function=SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
collection_name="my_collection"
vectorstore=Chroma.from_documents(collection_name=collection_name, documents=splits, embedding=embedding_function, persist_directory='./chroma_db')

print("Vector store created and persisted to './chroma_db")

Vector store created and persisted to './chroma_db


4: Check for similarity search and retrieve the context

In [18]:
# Perform similarity search

query="What is the Return Policy"
search_result= vectorstore.similarity_search(query, k=2)


print(f"\n Top 2 most relevant chunks for the query: '{query}' \n")
for i, result in enumerate(search_result, 1):
    print(f"Result {i}:")
    print(f"SOurce: {result.metadata.get('source','Unknown')}")
    print(f"Content:{result.page_content}")
    print()


 Top 2 most relevant chunks for the query: 'What is the Return Policy' 

Result 1:
SOurce: C:\Users\HP\Desktop\Langchain\RAG\OfflineData\HR documents.docx
Content:---



[Document ID: DEV002]

Title: Error Handling

Content:

APIs return standard HTTP error codes. Example: 400 for bad requests, 401 for unauthorized, and 500 for server errors. Include error logs and correlation IDs when reporting API issues.



---



[Document ID: DEV003]

Title: Webhook Integration

Content:

Webhooks allow real-time event updates by sending POST requests to client-defined endpoints. Ensure the endpoint URL is secure (HTTPS) and responds with status 200 within 5 seconds to acknowledge the webhook.



---



[Document ID: PROD001]

Title: Return Policy

Content:

Products can be returned within 30 days of delivery if unused and in original packaging. Refunds are processed within 7 working days once the item is inspected. Shipping costs are non-refundable.



---



[Document ID: PROD002]

Title: Warra

In [19]:
retriever=vectorstore.as_retriever(search_kwargs={"k":2})
retriever.invoke("Printer Not Responding")

[Document(id='ad724834-e0d2-47ec-958f-22b489af18e5', metadata={'source': 'C:\\Users\\HP\\Desktop\\Langchain\\RAG\\OfflineData\\HR documents.docx'}, page_content='---\n\n\n\n[Document ID: KB004]\n\nTitle: Printer Not Responding\n\nContent:\n\nWhen a printer is not responding, check power cables, printer queue status, and network connectivity. Try reinstalling the printer driver if jobs are stuck. For network printers, ensure the printer IP is reachable via ping.\n\n\n\n---\n\n\n\n[Document ID: KB005]\n\nTitle: Software Installation Request\n\nContent:\n\nSoftware installation requests must be approved by the IT Manager. After approval, deployment can be done using SCCM or manually if the package is not in the library. Ensure licensing compliance is verified.\n\n\n\n---\n\n\n\n[Document ID: HR001]\n\nTitle: Leave Policy\n\nContent:\n\nEmployees are entitled to 18 annual leaves, 12 sick leaves, and 10 casual leaves per year. Leaves must be applied through the HR portal and approved by rep

In [20]:
from langchain_core.prompts import ChatPromptTemplate
template ="""Answer the question based on the following context:
{context}

Question:{question}

Answer:"""

prompt=ChatPromptTemplate.from_template(template)

Augmetation(query + context )

In [21]:
from langchain.schema.runnable import RunnablePassthrough
rag_chain=(
    {'context': retriever, 'question':RunnablePassthrough()} | prompt
)
rag_chain.invoke("Work From Home Policy")

ChatPromptValue(messages=[HumanMessage(content="Answer the question based on the following context:\n[Document(id='f4eca791-7833-4feb-baed-9c374e0c7faf', metadata={'source': 'C:\\\\Users\\\\HP\\\\Desktop\\\\Langchain\\\\RAG\\\\OfflineData\\\\HR documents.docx'}, page_content='---\\n\\n\\n\\n[Document ID: HR002]\\n\\nTitle: Work From Home Policy\\n\\nContent:\\n\\nEmployees may request work from home up to two days per week with manager approval. VPN access and attendance tracking via the HR system are mandatory. Ensure data privacy standards are followed while working remotely.\\n\\n\\n\\n---\\n\\n\\n\\n[Document ID: HR003]\\n\\nTitle: Expense Reimbursement\\n\\nContent:\\n\\nAll business-related expenses must be submitted within 30 days of the expense date using the Expense Management Portal. Receipts are required for claims above ₹500. Approvals are handled by the Finance department.\\n\\n\\n\\n---\\n\\n\\n\\n[Document ID: DEV001]\\n\\nTitle: API Authentication\\n\\nContent:\\n\\nAll

In [22]:
# To get only the page content without any meta data
def docs2str(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [23]:
rag_chain=(
    {"context": retriever | docs2str, "question": RunnablePassthrough()} | prompt
)
rag_chain.invoke("Work From Home Policy")

ChatPromptValue(messages=[HumanMessage(content='Answer the question based on the following context:\n---\n\n\n\n[Document ID: HR002]\n\nTitle: Work From Home Policy\n\nContent:\n\nEmployees may request work from home up to two days per week with manager approval. VPN access and attendance tracking via the HR system are mandatory. Ensure data privacy standards are followed while working remotely.\n\n\n\n---\n\n\n\n[Document ID: HR003]\n\nTitle: Expense Reimbursement\n\nContent:\n\nAll business-related expenses must be submitted within 30 days of the expense date using the Expense Management Portal. Receipts are required for claims above ₹500. Approvals are handled by the Finance department.\n\n\n\n---\n\n\n\n[Document ID: DEV001]\n\nTitle: API Authentication\n\nContent:\n\nAll API requests must include a valid Bearer token obtained via the /auth/token endpoint. Tokens expire after 60 minutes and must be refreshed using the refresh token endpoint.\n\n\n\n---\n\n\n\n[Document ID: DEV002]\

Genaration- Output of the query

In [24]:
rag_chain=(
    {"context": retriever | docs2str, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
question="Work From Home Policy"
response=rag_chain.invoke(question)
print(response)

Employees may request to work from home up to two days per week with manager approval. VPN access and attendance tracking via the HR system are mandatory. It is important to ensure that data privacy standards are followed while working remotely.


Conversational RAG

Handling Follow Up Questions

In [25]:
# Example Conversation

from langchain_core.messages import HumanMessage, AIMessage
chat_history=[]
chat_history.extend([
    HumanMessage(content=question),
    AIMessage(content=response)
])
chat_history

[HumanMessage(content='Work From Home Policy', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Employees may request to work from home up to two days per week with manager approval. VPN access and attendance tracking via the HR system are mandatory. It is important to ensure that data privacy standards are followed while working remotely.', additional_kwargs={}, response_metadata={})]

a. contextualize prompt

In [26]:
from langchain_core.prompts import MessagesPlaceholder
contextualize_q_system_prompt=(
    "Given a chat history and the latest user question"
    "which might reference context in the chat history,"
    "formulate a standalone question which can be understood"
    "without the chat history. Do NOT answer the question,"
    "just reformulate it if needed and otherwise return it as is"
)

contextualize_q_prompt=ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}")
    ]
)

contextualize_chain=contextualize_q_prompt | llm | StrOutputParser()
contextualize_chain.invoke({"input": "why VPN access is mandatory" , "chat_history": chat_history })

'Why is VPN access required for remote work?'

b. retrieve contextualize prompt content from chroma

In [27]:
from langchain.chains import create_history_aware_retriever
history_aware_retriever= create_history_aware_retriever(llm, retriever, contextualize_q_prompt)
history_aware_retriever.invoke({"input": "why VPN access is mandatory" , "chat_history": chat_history })

[Document(id='321fd8e6-c894-4a9d-886c-51599a79db1d', metadata={'source': 'C:\\Users\\HP\\Desktop\\Langchain\\RAG\\OfflineData\\HR documents.docx'}, page_content='[Document ID: KB001]\n\nTitle: Unable to connect to VPN\n\nContent:\n\nIf users cannot connect to VPN, verify their credentials, check if the VPN service is running, and ensure network ports 443 and 500 are open. If the issue persists, restart the VPN client or check the firewall settings.\n\n\n\n---\n\n\n\n[Document ID: KB002]\n\nTitle: Resetting Active Directory Password\n\nContent:\n\nTo reset a user’s Active Directory password, open Active Directory Users and Computers, right-click the user account, and select "Reset Password." Ensure “User must change password at next logon” is checked for compliance.\n\n\n\n---\n\n\n\n[Document ID: KB003]\n\nTitle: DNS Resolution Issue\n\nContent:\n\nIf DNS resolution fails, verify the DNS server IP configuration in network adapter settings. Use the nslookup command to test name resoluti

In [28]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

qa_prompt=ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant Use the following context to answer the user's question."),
    ("system", "Context: {context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

question_answer_chain= create_stuff_documents_chain(llm, qa_prompt)
rag_chain=create_retrieval_chain(history_aware_retriever, question_answer_chain)
rag_chain.invoke({"input": "why VPN access is mandatory" , "chat_history": chat_history })

{'input': 'why VPN access is mandatory',
 'chat_history': [HumanMessage(content='Work From Home Policy', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Employees may request to work from home up to two days per week with manager approval. VPN access and attendance tracking via the HR system are mandatory. It is important to ensure that data privacy standards are followed while working remotely.', additional_kwargs={}, response_metadata={})],
 'context': [Document(id='321fd8e6-c894-4a9d-886c-51599a79db1d', metadata={'source': 'C:\\Users\\HP\\Desktop\\Langchain\\RAG\\OfflineData\\HR documents.docx'}, page_content='[Document ID: KB001]\n\nTitle: Unable to connect to VPN\n\nContent:\n\nIf users cannot connect to VPN, verify their credentials, check if the VPN service is running, and ensure network ports 443 and 500 are open. If the issue persists, restart the VPN client or check the firewall settings.\n\n\n\n---\n\n\n\n[Document ID: KB002]\n\nTitle: Resetting Active Dire

Managing conversation history using a database table

Building Multi User Chatbot

In [29]:
import sqlite3
from datetime import datetime

DB_NAME='rag_app.db'

def get_db_connection():
    conn=sqlite3.connect(DB_NAME)
    conn.row_factory=sqlite3.Row
    return conn

def create_Application_logs():
    conn=get_db_connection()
    conn.execute('''CREATE TABLE IF NOT EXISTS appplication_logs
                 (id INTEGER PRIMARY KEY AUTOINCREMENT,
                 session_id TEXT,
                 user_query TEXT,
                 gpt_response TEXT,
                 model TEXT,
                 created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)''')
    conn.close()

def insert_application_logs(session_id, user_query, gpt_response, model):
    conn=get_db_connection()
    conn.execute('INSERT INTO appplication_logs (session_id, user_query, gpt_response, model) VALUES (?,?,?,?)',
                 (session_id, user_query, gpt_response, model))
    conn.commit()
    conn.close()

def get_chat_history(session_id):
    conn= get_db_connection()
    cursor=conn.cursor()
    cursor.execute('SELECT user_query, gpt_response FROM appplication_logs WHERE session_id= ? ORDER BY created_at', (session_id,))
    messages=[]
    for row in cursor.fetchall():
        messages.extend([
            {"role": "human", "content": row['user_query']},
            {"role":"ai", "content": row['gpt_response']}
        ])
    conn.close()
    return messages

# Initialize the database
create_Application_logs()



In [32]:
import uuid
session_id= str(uuid.uuid4())
chat_history=get_chat_history(session_id)
print(chat_history)
question1="Work From Home Policy"
answer1=rag_chain.invoke({"input": question1 , "chat_history": chat_history })['answer']
insert_application_logs(session_id, question1, answer1, "gpt-4-o-mini")
print(f"Human: {question1}")
print(f"AI: {answer1} \n")

[]
Human: Work From Home Policy
AI: The Work From Home Policy allows employees to request to work from home up to two days per week, provided they have manager approval. Key points of the policy include:

- VPN access is required while working remotely.
- Attendance must be tracked through the HR system.
- Employees must ensure that data privacy standards are followed during remote work. 



In [33]:
question2="Why VPN access is mandatory"
chat_history=get_chat_history(session_id)
print(chat_history)
answer2=rag_chain.invoke({"input": question2 , "chat_history": chat_history })['answer']
insert_application_logs(session_id, question2, answer2, "gpt-4-o-mini")
print(f"Human: {question2}")
print(f"AI: {answer2} \n")

[{'role': 'human', 'content': 'Work From Home Policy'}, {'role': 'ai', 'content': 'The Work From Home Policy allows employees to request to work from home up to two days per week, provided they have manager approval. Key points of the policy include:\n\n- VPN access is required while working remotely.\n- Attendance must be tracked through the HR system.\n- Employees must ensure that data privacy standards are followed during remote work.'}]
Human: Why VPN access is mandatory
AI: VPN access is mandatory for several reasons:

1. **Security**: VPNs encrypt the data transmitted over the internet, protecting sensitive information from potential interception by unauthorized parties.

2. **Remote Access**: VPNs allow employees to securely connect to the company's internal network from remote locations, enabling access to necessary resources and applications.

3. **Data Privacy**: Using a VPN helps ensure that data remains confidential and secure, especially when accessing the network over pub

New User

In [34]:
session_id= str(uuid.uuid4())
question="Unable to connect to VPN"
chat_history=get_chat_history(session_id)
print(chat_history)
answer=rag_chain.invoke({"input": question , "chat_history": chat_history })['answer']
insert_application_logs(session_id, question, answer, "gpt-4-o-mini")
print(f"Human: {question}")
print(f"AI: {answer} \n")

[]
Human: Unable to connect to VPN
AI: If you are unable to connect to the VPN, please follow these steps:

1. **Verify Credentials**: Ensure that you are using the correct username and password for the VPN.

2. **Check VPN Service**: Confirm that the VPN service is running on your device.

3. **Open Network Ports**: Make sure that network ports 443 and 500 are open on your firewall and network settings.

4. **Restart VPN Client**: If the issue persists, try restarting the VPN client application.

5. **Check Firewall Settings**: Ensure that your firewall settings are not blocking the VPN connection.

If you continue to experience issues after trying these steps, consider reaching out to your IT support team for further assistance. 

